In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm  # for progress bars
from asian_option import GeometricAsianOption, ArithmeticAsianOption
from monte_carlo import price_asian_mc

# Set random seed for reproducibility
np.random.seed(42)

# Test parameters
S0 = 100.0  # Initial stock price
K = 100.0   # Strike price
T = 1.0     # Time to maturity (1 year)
r = 0.05    # Risk-free rate
sigma = 0.2 # Volatility
n_steps = 252  # Daily monitoring (252 trading days)

# Create instances of both option types
geometric = GeometricAsianOption(S0, K, T, r, sigma, n_steps)
arithmetic = ArithmeticAsianOption(S0, K, T, r, sigma, n_steps)

# Calculate geometric Asian option price (analytical)
geometric_price = geometric.price()
print(f"Geometric Asian Option Price (Analytical): {geometric_price:.4f}")

# Test Monte Carlo pricing with different numbers of simulations
n_sims_list = [1000, 5000, 10000, 50000, 100000]
results = []

for n_sims in tqdm(n_sims_list):
    # Price without control variate
    price_no_cv = price_asian_mc(arithmetic, n_sims, control_variate=False)
    
    # Price with control variate
    price_with_cv = price_asian_mc(arithmetic, n_sims, control_variate=True)
    
    results.append({
        'n_sims': n_sims,
        'no_cv': price_no_cv,
        'with_cv': price_with_cv
    })

# Plot convergence
plt.figure(figsize=(10, 6))
plt.plot([r['n_sims'] for r in results], [r['no_cv'] for r in results], 
         'b-o', label='Without Control Variate')
plt.plot([r['n_sims'] for r in results], [r['with_cv'] for r in results], 
         'r-o', label='With Control Variate')
plt.axhline(y=geometric_price, color='g', linestyle='--', 
            label='Geometric Price (Analytical)')
plt.xscale('log')
plt.xlabel('Number of Simulations')
plt.ylabel('Option Price')
plt.title('Convergence of Asian Option Prices')
plt.legend()
plt.grid(True)
plt.show()

# Print final results
print("\nFinal Results:")
print(f"{'N Sims':>10} {'No CV':>12} {'With CV':>12}")
print("-" * 35)
for r in results:
    print(f"{r['n_sims']:10d} {r['no_cv']:12.4f} {r['with_cv']:12.4f}")

# Calculate standard errors for the last simulation
n_sims = n_sims_list[-1]
n_trials = 10
prices_no_cv = []
prices_with_cv = []

print("\nCalculating standard errors...")
for _ in tqdm(range(n_trials)):
    prices_no_cv.append(price_asian_mc(arithmetic, n_sims, control_variate=False))
    prices_with_cv.append(price_asian_mc(arithmetic, n_sims, control_variate=True))

std_no_cv = np.std(prices_no_cv)
std_with_cv = np.std(prices_with_cv)

print(f"\nFor {n_sims} simulations:")
print(f"Standard Error (No CV): {std_no_cv:.6f}")
print(f"Standard Error (With CV): {std_with_cv:.6f}")
print(f"Variance Reduction: {(std_no_cv/std_with_cv)**2:.2f}x")